# Diabetes Prediction Using Logistic Regression

**Course:** Predictive Analytics &nbsp;|&nbsp; **Section:** D
**Student:** Anvay Sachin Sanil (2427030150)

**Goal:** Predict whether a patient has diabetes (`Outcome = 1`) from basic medical measurements,
using a Logistic Regression classifier.

**Dataset:** [Pima Indians Diabetes Database (Kaggle)](https://www.kaggle.com/datasets/uciml/pima-indians-diabetes-database)
768 female patients, 8 input features, 1 binary target.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, ConfusionMatrixDisplay,
                             roc_auc_score, RocCurveDisplay)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42

## 2. Load the data

In [ ]:
df = pd.read_csv("data/diabetes.csv")
print("Shape:", df.shape)
df.head()

In [ ]:
df.info()

In [ ]:
df.describe().T

## 3. Target distribution
About 35% of patients are diabetic, so the classes are somewhat imbalanced.

In [ ]:
print(df["Outcome"].value_counts())
sns.countplot(x="Outcome", data=df)
plt.title("Outcome (0 = No diabetes, 1 = Diabetes)")
plt.show()

## 4. Data cleaning
Some columns contain `0`, which is impossible for a living patient (for example, Glucose = 0 or BMI = 0).
These zeros are really missing values. We mark them as `NaN` and later fill them with the **median of the training set**.

In [ ]:
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
print("Zero counts before cleaning:")
print((df[zero_as_missing] == 0).sum())

df[zero_as_missing] = df[zero_as_missing].replace(0, np.nan)

## 5. Exploratory data analysis

In [ ]:
df.drop(columns="Outcome").hist(figsize=(12, 8), bins=20)
plt.suptitle("Feature distributions")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, col in zip(axes, ["Glucose", "BMI", "Age"]):
    sns.boxplot(x="Outcome", y=col, data=df, ax=ax)
    ax.set_title(f"{col} by Outcome")
plt.tight_layout()
plt.show()

## 6. Train / test split, imputation and scaling
- 80% training, 20% testing, stratified so both sets keep the same diabetic ratio.
- Missing values are filled with training-set medians (so no information leaks from the test set).
- Features are standardised, because Logistic Regression works best when features are on the same scale.

In [ ]:
X = df.drop(columns="Outcome")
y = df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

medians = X_train.median()
X_train = X_train.fillna(medians)
X_test = X_test.fillna(medians)

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s = scaler.transform(X_test)

print("Train:", X_train.shape, " Test:", X_test.shape)

## 7. Train the Logistic Regression model

In [ ]:
model = LogisticRegression(max_iter=1000)
model.fit(X_train_s, y_train)

y_pred = model.predict(X_test_s)
y_prob = model.predict_proba(X_test_s)[:, 1]

## 8. Evaluate the model

In [ ]:
print(f"Accuracy : {accuracy_score(y_test, y_pred):.3f}")
print(f"ROC-AUC  : {roc_auc_score(y_test, y_prob):.3f}")
print()
print(classification_report(y_test, y_pred, target_names=["No diabetes", "Diabetes"]))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred),
                       display_labels=["No diabetes", "Diabetes"]).plot(ax=axes[0], cmap="Blues")
axes[0].set_title("Confusion matrix")
RocCurveDisplay.from_predictions(y_test, y_prob, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC curve")
plt.tight_layout()
plt.show()

## 9. Which features matter most?
Each coefficient shows how a one-standard-deviation increase in a feature changes the log-odds of diabetes.
The odds ratio (`exp(coef)`) is easier to read: a value above 1 means higher risk.

In [ ]:
coef = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0],
    "Odds ratio": np.exp(model.coef_[0]),
}).sort_values("Coefficient", ascending=False)
print(coef.to_string(index=False))

sns.barplot(x="Coefficient", y="Feature", data=coef, color="steelblue")
plt.title("Logistic Regression coefficients (standardised features)")
plt.show()

## 10. Predict for a new patient

In [ ]:
new_patient = pd.DataFrame([{
    "Pregnancies": 2, "Glucose": 150, "BloodPressure": 75, "SkinThickness": 30,
    "Insulin": 120, "BMI": 33.5, "DiabetesPedigreeFunction": 0.6, "Age": 45,
}])
prob = model.predict_proba(scaler.transform(new_patient))[0, 1]
print(f"Probability of diabetes: {prob:.1%}")
print("Prediction:", "Diabetes" if prob >= 0.5 else "No diabetes")

## 11. Conclusion
- Logistic Regression gives a simple, interpretable baseline for diabetes screening.
- **Glucose** is by far the strongest predictor, followed by **BMI**, **Pregnancies** and **DiabetesPedigreeFunction**.
- Recall for the diabetic class is lower than for the non-diabetic class, so the model misses some diabetic patients.
  Lowering the decision threshold below 0.5 would catch more of them at the cost of more false alarms.
- Possible improvements: tune the threshold, try class weights, or compare with tree-based models.